In [14]:
import os
import json
import boto3
from dotenv import load_dotenv
from pymongo import MongoClient

# Load environment variables
load_dotenv()

# MongoDB connection details
MONGODB_URI = os.getenv('MONGODB_URI_REALM')
DB_NAME = 'gpt'
COLLECTION_NAME = 'list'

# Cloudflare R2 bucket details
R2_ACCESS_KEY = os.getenv('CLOUDFLARE_ACCESS_KEY')
R2_SECRET_KEY = os.getenv('CLOUDFLARE_SECRET_KEY')
R2_BUCKET = os.getenv('CLOUDFLARE_BUCKET')
R2_ENDPOINT = os.getenv('CLOUDFLARE_ENDPOINT')

# Connect to MongoDB
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Helper function to upload to Cloudflare R2
def upload_to_r2(local_file_path, r2_key):
    session = boto3.session.Session()
    s3 = session.client(
        's3',
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    print(f"Uploading {local_file_path} to R2 as {r2_key}")
    if os.path.exists(local_file_path) and os.path.getsize(local_file_path) > 0:
        try:
            s3.upload_file(local_file_path, R2_BUCKET, r2_key)
            print(f"Successfully uploaded {local_file_path} to R2 as {r2_key}")
        except Exception as e:
            print(f"Error uploading {local_file_path}: {str(e)}")
    else:
        print(f"Error: {local_file_path} does not exist or is empty.")

# Aggregation to group by postid and create JSON files
postid_aggregation = [
    {"$group": {
        "_id": "$postid",
        "docs": {
            "$push": {
                "_id": {"$toString": "$_id"},  # Convert ObjectId to string
                "title": "$title",
                "slug": "$slug",
                "excerpt": "$excerpt",
                "outline": "$outline",
                "exists": {"$cond": {"if": {"$ifNull": ["$content", False]}, "then": True, "else": False}}
            }
        }
    }},
]

# postid_documents = list(collection.aggregate(postid_aggregation))

# # Save each grouped postid as a JSON file and upload to Cloudflare R2
# os.makedirs("feed", exist_ok=True)
# for post in postid_documents:
#     postid = post["_id"]
#     local_file_path = f"feed/{postid}.json"
#     with open(local_file_path, "w") as f:
#         json.dump(post["docs"], f, indent=4)
#     upload_to_r2(local_file_path, f"feed/{postid}.json")



In [15]:
category_aggregation = [
    # Group documents by category
    {"$group": {
        "_id": "$category",
        "docs": {
            "$push": {
                "_id": {"$toString": "$_id"},  # Convert ObjectId to string
                "title": "$title",
                "slug": "$slug",
                "excerpt": "$excerpt",
                "outline": "$outline",
                "exists": {"$cond": {"if": {"$ifNull": ["$content", False]}, "then": True, "else": False}}
            }
        }
    }},
    # Filter to ensure categories have at least 10 documents
    {"$match": {"docs.10": {"$exists": True}}},
    # Unwind docs to apply sampling per document level
    {"$unwind": "$docs"},
    # Sample 100 documents randomly from each category
    {"$sample": {"size": 100}},
    # Regroup sampled documents by category
    {"$group": {
        "_id": "$_id",
        "docs": {"$push": "$docs"}
    }}
]

category_documents = list(collection.aggregate(category_aggregation))

# Save each grouped category as a JSON file and upload to Cloudflare R2
for category in category_documents:
    category_name = category["_id"]
    local_file_path = f"feed/{category_name}.json"
    with open(local_file_path, "w") as f:
        json.dump(category["docs"], f, indent=4)
    upload_to_r2(local_file_path, f"feed/{category_name}.json")

Uploading feed/Healthy Eating.json to R2 as feed/Healthy Eating.json
Successfully uploaded feed/Healthy Eating.json to R2 as feed/Healthy Eating.json
Uploading feed/food.json to R2 as feed/food.json
Successfully uploaded feed/food.json to R2 as feed/food.json
Uploading feed/travel.json to R2 as feed/travel.json
Successfully uploaded feed/travel.json to R2 as feed/travel.json
Uploading feed/love.json to R2 as feed/love.json
Successfully uploaded feed/love.json to R2 as feed/love.json
